# 02 Baselines: median and matrix pricing

Role 2. Due Thursday 19 Nov: every other owner compares against these numbers.

Three baselines, all from `src/ml2/baseline.py` (one definition for everyone, because the quality gate depends on it). Open that file and read it before running this. If something in it looks wrong, tell Miguel; do not patch a copy here.

Class notebooks followed: `Regression_Improve Bagg & RF with Boosting` ("0. Baseline model: median value").

Every step is coded and runs top to bottom on the shared panel (about 2 minutes). Your job: run it, read each output,
write the interpretation cells in your own words, and be able to explain every line (the professor can ask
anyone). Where a step says **Your reading**, replace the prompt with what the output shows.

Rules: validation only (`partition == "valid"`); test is opened once, in `99_final`. Shared code comes from `ml2`
and is never copied here. Figures are saved to `reports/figures/02_*.png`. Paste the lines printed at the
end into `RUN_LOG.md`. Clear the outputs before you commit (the data is licensed: no rows in Git).

## 1. Setup

In [ ]:
import pandas as pd
import matplotlib.pyplot as plt

from ml2.baseline import fit_matrix_frozen, matrix_same_date, median_baseline, predict_matrix_frozen
from ml2.config import load_config, resolve
from ml2.data import check_panel, load_panel
from ml2.evaluate import metrics_by_stratum

cfg = load_config()
TARGET = cfg["problem"]["target"]
FIG = resolve("reports/figures")
FIG.mkdir(parents=True, exist_ok=True)
MATURITY_ORDER = ["1-3y", "3-5y", "5-7y", "7-10y", "10-20y", "20y+"]

check_panel()
df = load_panel(partitions=["train", "valid"])
train = df[df.partition == "train"]
valid = df[df.partition == "valid"].copy()
print(f"train {len(train):,} rows, valid {len(valid):,} rows")

## 2. Class baseline: the median

As in the class notebook ("0. Baseline model: median value"): predict the median training spread for every bond.
One number for everything, so it ignores grade and maturity.

In [ ]:
valid["pred_median"] = median_baseline(train, valid)
print(f"median training spread: {valid['pred_median'].iloc[0]:.1f} bp")
metrics_by_stratum(valid, TARGET, "pred_median", "grade_bucket").round(1)

## 3. Matrix pricing, frozen

The pricing desk's rule: the median spread of bonds in the same grade x maturity bucket x sector cell, falling
back to grade x maturity, then grade, when a cell has fewer than 20 bonds. Frozen: the medians come from the
training months only. Reference row.

In [ ]:
valid["pred_matrix_frozen"] = predict_matrix_frozen(fit_matrix_frozen(train), valid)
metrics_by_stratum(valid, TARGET, "pred_matrix_frozen", "grade_bucket").round(1)

## 4. Matrix pricing, re-marked each date

What a desk actually does: on each snapshot date, a bond is priced from the same-date spreads of **other
companies'** bonds in its cell (its own company is left out, so it never prices itself). This is the gate
baseline: the final model must beat it by at least 10% in IG and in HY. About 1.5 minutes.

In [ ]:
valid["pred_matrix_same_date"] = matrix_same_date(valid)
metrics_by_stratum(valid, TARGET, "pred_matrix_same_date", "grade_bucket").round(1)

## 5. Baseline numbers

All three, by grade and by maturity bucket.

In [ ]:
BASELINES = {
    "pred_median": "Median spread (class baseline)",
    "pred_matrix_frozen": "Matrix pricing, frozen at the training months",
    "pred_matrix_same_date": "Matrix pricing, re-marked each date (gate baseline)",
}
by_grade = pd.concat({name: metrics_by_stratum(valid, TARGET, col, "grade_bucket").set_index("stratum")
                      for col, name in BASELINES.items()})
by_grade[["n", "mae_bps", "rmse_bps", "medae_bps", "r2"]].round(2)

In [ ]:
by_maturity = pd.DataFrame({
    name: metrics_by_stratum(valid, TARGET, col, "maturity_bucket").set_index("stratum")["mae_bps"]
    for col, name in BASELINES.items()
}).loc[MATURITY_ORDER]

fig, ax = plt.subplots(figsize=(9, 4))
by_maturity.plot.bar(ax=ax)
ax.set_xlabel("Maturity bucket"); ax.set_ylabel("MAE on valid (bp)")
ax.set_title("Baseline error by maturity, valid"); ax.legend(fontsize=8)
fig.savefig(FIG / "02_mae_by_maturity.png", dpi=150, bbox_inches="tight")
by_maturity.round(1)

**Your reading.** Why does re-marking beat freezing? (The market moved after May 2025, and the frozen medians do not know it.) Why is HY RMSE far above its MAE? (A few very wide spreads.)

## 6. Hand-off and RUN_LOG

Post the three valid MAEs (IG, HY, all) to the team and paste these lines into `RUN_LOG.md`. The figure for slide 7
is `reports/figures/02_mae_by_maturity.png`.

In [ ]:
today = pd.Timestamp.today().strftime("%Y-%m-%d")
for name in BASELINES.values():
    m = by_grade.loc[name, "mae_bps"]
    print(f"| {today} | <your name> | 02_baseline_matrix_pricing | {name} | grade, maturity, sector | "
          f"fit on train, scored on valid | {m['IG']:.1f} / {m['HY']:.1f} | all: {m['ALL']:.1f} |")